# Espaços de busca dos modelos de Ensemble — Random Forest e Gradient Boosting (#186)

Este notebook documenta e justifica os espaços de busca aleatória (`RandomizedSearchCV`) do Random Forest e do Gradient Boosting, definidos em `src/ensembles.py`. Ele **não executa nenhuma busca**: treinar as combinações nos folds do #102 e comparar contra os pisos fica para os cards que dependem deste — #187 (Random Forest), #188 e #189 (Gradient Boosting) e #190.

**Pontos de partida em comum com o candidato único da seção 4.3.** A base analítica tem 484.915 registros e 20,44% de Detratores (seção 4.2.1 da documentação); o treino usado pela validação cruzada por Cliente do #102, depois do split, tem 341.962 linhas. O contrato de features do card 01 (`FEATURE_SET_V1`, em `scripts/preprocessamento_nps.py`) declara 11 features brutas, e é o mesmo contrato usado aqui.

**Gradient Boosting: duas bibliotecas, por enquanto.** O #185 ainda não decidiu entre `HistGradientBoostingClassifier` (scikit-learn) e `XGBClassifier` (XGBoost) para o trabalho de Ensemble, então este notebook e `src/ensembles.py` mantêm os dois espaços lado a lado. No dia em que o #185 decidir, a biblioteca descartada sai dos dois lugares no mesmo commit.

**Os testes que garantem que os espaços funcionam** ficam em `tests/test_ensembles.py`, e não neste notebook: eles sorteiam 40 combinações de cada espaço com semente fixa e instanciam o estimador correspondente, o que é rápido demais para justificar rodar aqui a cada execução do notebook. Este notebook sorteia só uma amostra pequena, para inspeção visual dos valores.

## 1. Random Forest

### 1.1. Espaço de busca declarado

A célula abaixo importa `ESPACO_RANDOM_FOREST` de `src/ensembles.py` e lista os cinco eixos declarados. A importação do módulo não ajusta nenhum estimador (CR01): o que a célula imprime é a distribuição em si, não uma amostra dela.

In [ ]:
import sys
from pathlib import Path

for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

from ensembles import ESPACO_RANDOM_FOREST


def descrever(distribuicao):
    """Nome e parâmetros da distribuição, em vez do endereço de memória do repr padrão.

    O repr padrão de um objeto `scipy.stats` traz o endereço em memória, que muda
    a cada execução e sujaria o diff do notebook sem nenhuma informação nova.
    """
    if isinstance(distribuicao, list):
        return f"lista com {len(distribuicao)} opções: {distribuicao}"
    return f"scipy.stats.{distribuicao.dist.name}, parâmetros={distribuicao.args or distribuicao.kwds}"


for nome, distribuicao in ESPACO_RANDOM_FOREST.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** Cinco eixos: `n_estimators`, `max_depth` e `min_samples_leaf` são inteiros sorteados por `scipy.stats.randint`; `max_features` é uma fração contínua sorteada por `scipy.stats.uniform`; `class_weight` é a única lista simples do espaço, porque reponderação de classe é uma escolha categórica, sem escala, e não uma distribuição numérica. A tabela consolidada da seção 3 traz o intervalo e a justificativa de cada um.

### 1.2. Amostra de combinações, para inspeção visual

`ParameterSampler` sorteia cinco combinações do espaço com semente fixa. Cinco é o suficiente para olhar os valores a olho nu; a garantia de que 40 combinações são aceitas pelo `RandomForestClassifier` é o que `tests/test_ensembles.py` cobre, não esta célula.

In [ ]:
import pandas as pd
from sklearn.model_selection import ParameterSampler

SEMENTE = 42

amostra_rf = pd.DataFrame(
    ParameterSampler(ESPACO_RANDOM_FOREST, n_iter=5, random_state=SEMENTE)
)
amostra_rf

**Leitura da tabela.** Cada linha é uma combinação que um `RandomizedSearchCV` real sortearia e passaria ao `RandomForestClassifier`. `max_features` sai como fração entre 0,3 e 1,0, nunca uma contagem de colunas; `class_weight` alterna entre as três opções declaradas. Os valores concretos mudam se a semente mudar, mas a mesma semente reproduz sempre esta tabela, como `test_amostragem_e_reprodutivel_com_a_mesma_semente` confere.

## 2. Gradient Boosting

Duas bibliotecas convivem aqui até o #185 decidir. As duas seções abaixo seguem a mesma estrutura da seção 1: espaço declarado, depois amostra pequena para inspeção.

### 2.1. `HistGradientBoostingClassifier` (scikit-learn)

Mesma biblioteca do candidato único da seção 4.3 do notebook `modelagem.ipynb`. A célula lista `ESPACO_GRADIENT_BOOSTING_HISTGB`.

In [ ]:
from ensembles import ESPACO_GRADIENT_BOOSTING_HISTGB

for nome, distribuicao in ESPACO_GRADIENT_BOOSTING_HISTGB.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** Cinco eixos: `learning_rate` é log-uniforme, porque o efeito do passo sobre o número de iterações necessário é multiplicativo; `max_leaf_nodes`, `max_iter` e `min_samples_leaf` são inteiros; `class_weight` é lista, pelo mesmo motivo da seção 1. `early_stopping` não entra no espaço — continua fixo em `False` em `criar_candidato` (`src/modelo.py`), porque ligá-lo vazaria Cliente entre ajuste e medição interna, e isso não é uma escolha que a busca deva reabrir.

### 2.2. `XGBClassifier` (XGBoost)

A célula lista `ESPACO_GRADIENT_BOOSTING_XGBOOST`. Sete eixos, dois a mais que o espaço do HistGB, porque o XGBoost expõe sub-amostragem de linhas e de colunas como parâmetros próprios, que o HistGB não tem por discretizar em histogramas em vez de amostrar.

In [ ]:
from ensembles import ESPACO_GRADIENT_BOOSTING_XGBOOST

for nome, distribuicao in ESPACO_GRADIENT_BOOSTING_XGBOOST.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** `learning_rate` usa a mesma distribuição log-uniforme do HistGB, para o custo por ajuste do #185 comparar as duas bibliotecas sob o mesmo raciocínio de passo. `scale_pos_weight` é o único eixo com teto calculado a partir da base: `RAZAO_DESBALANCEAMENTO` (~3,89) é a proporção de não-Detratores por Detrator, e um `scale_pos_weight` nesse valor neutraliza o desbalanceamento por completo.

### 2.3. Amostra combinada, para inspeção visual

Uma amostra pequena de cada biblioteca, lado a lado.

In [ ]:
amostra_histgb = pd.DataFrame(
    ParameterSampler(ESPACO_GRADIENT_BOOSTING_HISTGB, n_iter=5, random_state=SEMENTE)
)
amostra_xgboost = pd.DataFrame(
    ParameterSampler(ESPACO_GRADIENT_BOOSTING_XGBOOST, n_iter=5, random_state=SEMENTE)
)
print("HistGradientBoostingClassifier:")
display(amostra_histgb)
print("XGBClassifier:")
display(amostra_xgboost)

**Leitura das tabelas.** As duas amostram a mesma semente e o mesmo `n_iter`, mas eixos diferentes: `max_leaf_nodes` de um lado, `max_depth` do outro, porque cada biblioteca limita o tamanho da árvore à sua maneira. Nenhuma das duas tabelas depende da outra ter rodado antes — os dois espaços são independentes, e é isso que permite descartar um deles no dia em que o #185 decidir sem tocar no que sobra.

## 3. Tabela consolidada de hiperparâmetros (CR04)

| Modelo | Hiperparâmetro | Distribuição | Intervalo | Justificativa |
|---|---|---|---|---|
| Random Forest | `n_estimators` | `randint` | 200 a 600 | Com 341.962 linhas de treino a variância de cada árvore já é pequena; acima de algumas centenas o ganho de agregar mais árvores fica marginal e o custo segue linear no número delas. |
| Random Forest | `max_depth` | `randint` | 3 a 20 | Teto próximo de log₂(341.962) ≈ 18,4: uma árvore balanceada raramente precisa de mais níveis para separar o treino inteiro. Profundidade ilimitada arriscaria árvores memorizando Cliente, o mesmo risco documentado para `min_samples_leaf` no #103. |
| Random Forest | `min_samples_leaf` | `randint` | 1 a 100 | 100 é o valor que o #103 escolheu para o Gradient Boosting nesta mesma base; o piso em 1 mantém a ponta não regularizada disponível para a busca comparar. |
| Random Forest | `max_features` | `uniform` | 0,3 a 1,0 (fração) | O contrato do card 01 declara 11 features brutas; `"sqrt"`/`"log2"` datam de bases com centenas de colunas e amostrariam ~3 delas por split, descartando a maior parte do sinal disponível. |
| Random Forest | `class_weight` | lista | `balanced`, `balanced_subsample`, `None` | 20,44% de detração é desbalanceamento moderado. Ao contrário do candidato único (onde reponderar prejudicaria a probabilidade calibrada do #103), o Random Forest vota por árvore e não expõe a mesma dependência direta da calibração — a busca decide empiricamente qual das três funciona melhor nesta base. |
| Gradient Boosting (HistGB) | `learning_rate` | `loguniform` | 0,01 a 0,3 | O #103 fixou 0,05 e a grade do #104 testou 0,05/0,10; escala log porque o efeito do passo sobre o número de iterações necessário é multiplicativo. |
| Gradient Boosting (HistGB) | `max_leaf_nodes` | `randint` | 15 a 127 | Cobre o padrão da biblioteca (31, usado pelo #103) e o ponto testado pela grade do #104 (63), com margem para os dois lados. |
| Gradient Boosting (HistGB) | `max_iter` | `randint` | 100 a 600 | Cobre os três pontos da grade do #104 (150, 300, 600). Sem parada antecipada, é o único limite do ensemble. |
| Gradient Boosting (HistGB) | `l2_regularization` | `uniform` | 0,0 a 2,0 | O #103 fixou 1,0 contra o padrão 0,0 da biblioteca; o intervalo cobre os dois extremos e o dobro da escolha do #103. |
| Gradient Boosting (HistGB) | `min_samples_leaf` | `randint` | 20 a 200 | Mesmo raciocínio do Random Forest: folha pequena sobre 341.962 linhas memoriza Cliente. O piso de 20 é o padrão da biblioteca. |
| Gradient Boosting (HistGB) | `class_weight` | lista | `None`, `balanced` | O #103 documenta por que `class_weight=None` foi a escolha do candidato: reponderar deslocaria a probabilidade predita, piorando o Brier da seção 6 e o limiar da seção 7. O eixo entra na busca para que a comparação empírica sustente essa escolha, não só o argumento teórico. |
| Gradient Boosting (XGBoost) | `learning_rate` | `loguniform` | 0,01 a 0,3 | Mesma faixa e mesma razão log-uniforme do HistGB, por serem as duas gradient boosting sobre árvores. |
| Gradient Boosting (XGBoost) | `max_depth` | `randint` | 3 a 10 | O XGBoost limita a árvore por profundidade, e não por número de folhas; 10 níveis já produzem até 2¹⁰ folhas, ordem de interação comparável ao teto de 127 folhas do espaço irmão. |
| Gradient Boosting (XGBoost) | `n_estimators` | `randint` | 100 a 600 | Mesmo intervalo de `max_iter` do HistGB, para que o custo por ajuste medido no #185 seja comparável entre as duas bibliotecas sob o mesmo orçamento de iterações. |
| Gradient Boosting (XGBoost) | `min_child_weight` | `randint` | 1 a 20 | Equivalente do XGBoost a `min_samples_leaf`: soma mínima de peso das observações numa folha antes dela poder ser criada, mesma lógica de regularização contra folha memorizando Cliente. |
| Gradient Boosting (XGBoost) | `reg_lambda` | `uniform` | 0,0 a 2,0 | Mesma faixa de `l2_regularization` do HistGB, pelo nome equivalente no XGBoost. |
| Gradient Boosting (XGBoost) | `subsample` | `uniform` | 0,6 a 1,0 | Mecanismo de regularização do XGBoost sem par direto no HistGB (que discretiza em histogramas em vez de amostrar linhas). O piso de 0,6 evita perder informação suficiente para exigir muito mais iterações. |
| Gradient Boosting (XGBoost) | `colsample_bytree` | `uniform` | 0,5 a 1,0 | Equivalente a `max_features` do Random Forest: com 11 features brutas no contrato do card 01, um piso baixo deixaria cada árvore enxergar poucas colunas. |
| Gradient Boosting (XGBoost) | `scale_pos_weight` | `uniform` | 1,0 a ~3,89 | O teto é `RAZAO_DESBALANCEAMENTO`, a proporção observada de não-Detratores por Detrator (385.775 / 99.140). O XGBoost não tem uma versão `"balanced"` pronta como o scikit-learn; este parâmetro multiplica o gradiente da classe positiva, e o teto no valor exato da proporção é o ponto em que a reponderação neutraliza o desbalanceamento por completo. |

**O que fica para os cards seguintes.** Este notebook não mede custo por ajuste nem escolhe `n_iter` para a `RandomizedSearchCV` real — isso depende do #185, que ainda está aberto. Quando ele decidir a biblioteca de Gradient Boosting, a seção 2 deste notebook e o espaço correspondente em `src/ensembles.py` perdem a metade descartada no mesmo commit.